# 05A LeRobot 官方 ACT 基线复现、训练、推理与源码解析

这份 Notebook 的目标不是只会运行：

```powershell
lerobot-train --policy.type=act
```

而是沿着 **LeRobot 0.6.1 的真实调用链**理解：

```text
PushT Dataset
    ↓
LeRobotDatasetMetadata
    ↓
ACTConfig
    ↓
make_policy()
    ↓
ACTPolicy
    ↓
ACT 神经网络
    ↓
forward() 训练损失
    ↓
optimizer 更新
    ↓
checkpoint
    ↓
from_pretrained()
    ↓
predict_action_chunk()
    ↓
select_action()
```

完成后应当掌握：

1. LeRobot 的 ACT 源码文件在哪里；
2. 每个核心类与函数负责什么；
3. PushT 数据如何变成 ACT 的 action chunk；
4. 官方训练器如何创建数据、模型、处理器和优化器；
5. `forward()`、`predict_action_chunk()`、`select_action()` 的区别；
6. 如何训练、保存、恢复和推理；
7. 如何加载社区训练好的 PushT ACT checkpoint 做对照；
8. 如何查看模型仓库与本地 checkpoint 的文件架构。

---

## 关于 PushT 预训练 ACT

截至本 Notebook 编写时：

- `lerobot/pusht` 是官方数据集；
- LeRobot 官方组织公开了 `lerobot/diffusion_pusht`、`lerobot/vqbet_pusht` 等 PushT checkpoint；
- 没有确认到官方组织名下的 `lerobot/act_pusht`；
- Hugging Face Hub 上存在社区训练好的 PushT ACT，例如：
  - `aadarshram/act_pusht`
  - `Lemon-03/ACT_PushT_test`
  - `vvrs/act-pusht`

本 Notebook 默认用：

```text
aadarshram/act_pusht
```

做预训练模型对照。它是社区模型，不应当当成官方基准。


# A. 运行前检查

请确认 VS Code 右上角 Kernel 为：

```text
Python (lerobot-win)
D:\Desktop\robot\envs\lerobot-win\python.exe
```

Notebook 中的长时间操作默认关闭：

```python
RUN_SMOKE_TRAIN = False
RUN_FULL_TRAIN = False
RUN_PRETRAINED_DOWNLOAD = False
RUN_CLOSED_LOOP_EVAL = False
```

理解完相应章节后，再手动改成 `True`。


In [1]:
from __future__ import annotations

import importlib.metadata
import inspect
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import torch
import lerobot

print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])
print("LeRobot:", importlib.metadata.version("lerobot"))
print("LeRobot package:", Path(lerobot.__file__).resolve())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA build:", torch.version.cuda)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_LEROBOT_HOME:", os.environ.get("HF_LEROBOT_HOME"))


Python: d:\Desktop\robot\envs\lerobot-win\python.exe
Python version: 3.12.13
LeRobot: 0.6.1
LeRobot package: D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\__init__.py
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA build: 12.8
HF_HOME: D:\Desktop\robot\datasets
HF_LEROBOT_HOME: D:\Desktop\robot\datasets\lerobot


# B. 找到 LeRobot ACT 的真实源码文件

你当前通过 pip/Conda 安装的 LeRobot 源码位于：

```text
<当前环境>\Lib\site-packages\lerobot
```

ACT 不是一个单独的 `.pt` 文件，而是一组配置、模型、工厂、处理器和训练脚本。


In [2]:
LEROBOT_ROOT = Path(lerobot.__file__).resolve().parent
ACT_ROOT = LEROBOT_ROOT / "policies" / "act"

print("LeRobot root:", LEROBOT_ROOT)
print("ACT root:", ACT_ROOT)
print("ACT目录存在:", ACT_ROOT.exists())


LeRobot root: D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot
ACT root: D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act
ACT目录存在: True


In [3]:
def print_tree(root: Path, max_depth: int = 3) -> None:
    # 打印文件树，只显示到指定深度
    root = root.resolve()
    print(root.name + "/")

    paths = sorted(root.rglob("*"), key=lambda p: (len(p.parts), str(p).lower()))
    for path in paths:
        relative = path.relative_to(root)

        if len(relative.parts) > max_depth:
            continue

        indent = "    " * (len(relative.parts) - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}├── {path.name}{suffix}")


print_tree(ACT_ROOT, max_depth=3)


act/
├── __init__.py
├── __pycache__/
├── configuration_act.py
├── modeling_act.py
├── processor_act.py
    ├── __init__.cpython-312.pyc
    ├── configuration_act.cpython-312.pyc
    ├── modeling_act.cpython-312.pyc
    ├── processor_act.cpython-312.pyc


ACT 目录通常至少应包含：

```text
lerobot/
└── policies/
    └── act/
        ├── __init__.py
        ├── configuration_act.py
        ├── modeling_act.py
        └── 可能存在的处理器或说明文件
```

不要依赖这里写死的名称；以上一格在你电脑上打印的实际结果为准。


In [4]:
RELEVANT_CANDIDATES = [
    LEROBOT_ROOT / "policies" / "act" / "configuration_act.py",
    LEROBOT_ROOT / "policies" / "act" / "modeling_act.py",
    LEROBOT_ROOT / "policies" / "act" / "processor_act.py",
    LEROBOT_ROOT / "policies" / "factory.py",
    LEROBOT_ROOT / "policies" / "pretrained.py",
    LEROBOT_ROOT / "scripts" / "lerobot_train.py",
    LEROBOT_ROOT / "scripts" / "lerobot_eval.py",
    LEROBOT_ROOT / "configs" / "train.py",
    LEROBOT_ROOT / "configs" / "eval.py",
    LEROBOT_ROOT / "datasets" / "lerobot_dataset.py",
    LEROBOT_ROOT / "datasets" / "factory.py",
    LEROBOT_ROOT / "utils" / "feature_utils.py",
]

print("与 ACT 训练/推理有关的源码文件：")
for path in RELEVANT_CANDIDATES:
    print("[存在]" if path.exists() else "[不存在]", path)


与 ACT 训练/推理有关的源码文件：
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\configuration_act.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\modeling_act.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\processor_act.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\factory.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\pretrained.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\scripts\lerobot_train.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\scripts\lerobot_eval.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\configs\train.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\configs\eval.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\datasets\lerobot_dataset.py
[存在] D:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\datasets\factory.py

## B1. 这些文件分别负责什么

```text
configuration_act.py
    ACTConfig：模型结构、时间窗口、优化器参数

modeling_act.py
    ACTPolicy：LeRobot策略接口
    ACT：真正的神经网络
    ACTEncoder / ACTDecoder：Transformer组件
    ACTTemporalEnsembler：推理时动作融合

policies/factory.py
    make_policy()
    make_pre_post_processors()
    把数据集特征转换成策略输入输出

scripts/lerobot_train.py
    官方训练入口
    创建Dataset、Policy、Processor、Optimizer
    执行forward/backward/save

scripts/lerobot_eval.py
    官方仿真评测入口

datasets/lerobot_dataset.py
    LeRobotDataset、LeRobotDatasetMetadata
    根据delta_timestamps读取时间窗口

utils/feature_utils.py
    dataset_to_policy_features()
```


# C. 源码阅读工具

下面的工具可以在 Notebook 中直接查看：

- 类或函数来自哪个文件；
- 源码从第几行开始；
- 函数签名；
- 实际 Python 源码。

这比只看文档更接近算法研究。


In [5]:
def describe_object(obj: Any) -> None:
    print("对象:", obj)
    print("模块:", getattr(obj, "__module__", None))

    try:
        print("源码文件:", inspect.getsourcefile(obj))
    except Exception as exc:
        print("源码文件读取失败:", exc)

    try:
        _, start_line = inspect.getsourcelines(obj)
        print("起始行:", start_line)
    except Exception as exc:
        print("起始行读取失败:", exc)

    try:
        print("签名:", inspect.signature(obj))
    except Exception as exc:
        print("签名读取失败:", exc)


def show_source(obj: Any, max_lines: int | None = 120) -> None:
    source = inspect.getsource(obj)
    lines = source.splitlines()

    if max_lines is not None and len(lines) > max_lines:
        total_lines = len(lines)
        lines = lines[:max_lines]
        lines.append(f"... 已截断，总行数约 {total_lines}")

    print("\n".join(lines))


# D. ACTConfig：配置如何决定数据窗口与模型结构


In [8]:
from lerobot.policies.act.configuration_act import ACTConfig

describe_object(ACTConfig)


对象: <class 'lerobot.policies.act.configuration_act.ACTConfig'>
模块: lerobot.policies.act.configuration_act
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\configuration_act.py
起始行: 22
签名: (n_obs_steps: int = 1, input_features: dict[str, lerobot.configs.types.PolicyFeature] | None = <factory>, output_features: dict[str, lerobot.configs.types.PolicyFeature] | None = <factory>, device: str | None = None, use_amp: bool = False, use_peft: bool = False, push_to_hub: bool = True, repo_id: str | None = None, private: bool | None = None, tags: list[str] | None = None, license: str | None = None, pretrained_path: pathlib.Path | None = None, pretrained_revision: str | None = None, chunk_size: int = 100, n_action_steps: int = 100, normalization_mapping: dict[str, lerobot.configs.types.NormalizationMode] = <factory>, vision_backbone: str = 'resnet18', pretrained_backbone_weights: str | None = 'ResNet18_Weights.IMAGENET1K_V1', replace_final_stride_with_dilation: int = F

In [7]:
config_source = inspect.getsource(ACTConfig)
print(config_source)


@PreTrainedConfig.register_subclass("act")
@dataclass
class ACTConfig(PreTrainedConfig):
    """Configuration class for the Action Chunking Transformers policy.

    Defaults are configured for training on bimanual Aloha tasks like "insertion" or "transfer".

    The parameters you will most likely need to change are the ones which depend on the environment / sensors.
    Those are: `input_features` and `output_features`.

    Notes on the inputs and outputs:
        - Either:
            - At least one key starting with "observation.image is required as an input.
              AND/OR
            - The key "observation.environment_state" is required as input.
        - If there are multiple keys beginning with "observation.images." they are treated as multiple camera
          views. Right now we only support all images having the same shape.
        - May optionally work without an "observation.state" key for the proprioceptive robot state.
        - "action" is required as an output 

阅读 `ACTConfig` 时重点寻找：

```text
chunk_size
    一次预测的动作数量

n_action_steps
    每次查询模型后实际消费多少个动作

observation_delta_indices
    Dataset读取哪些观测时刻

action_delta_indices
    Dataset读取哪些未来动作时刻

vision_backbone
    图像编码器，例如ResNet18

dim_model / n_heads / n_encoder_layers / n_decoder_layers
    Transformer规模

use_vae / latent_dim / kl_weight
    CVAE相关配置

optimizer_lr / optimizer_weight_decay / optimizer_lr_backbone
    优化器参数
```


In [9]:
demo_config = ACTConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    push_to_hub=False,
    chunk_size=100,
    n_action_steps=100,
)

print("action_delta_indices前20项:")
print(demo_config.action_delta_indices[:20])

print("action_delta_indices长度:")
print(len(demo_config.action_delta_indices))

print("observation_delta_indices:")
print(demo_config.observation_delta_indices)


action_delta_indices前20项:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
action_delta_indices长度:
100
observation_delta_indices:
None


# E. ACTPolicy 与 ACT 的分工

这是最容易混淆的两层。

```text
ACTPolicy
    LeRobot统一策略接口
    管理训练loss、动作队列、推理API、checkpoint

ACT
    真正的PyTorch神经网络
    ResNet + CVAE Encoder + Transformer + Action Head
```


In [10]:
from lerobot.policies.act.modeling_act import (
    ACT,
    ACTPolicy,
    ACTTemporalEnsembler,
    ACTEncoder,
    ACTEncoderLayer,
    ACTDecoder,
    ACTDecoderLayer,
)

for obj in [
    ACTPolicy,
    ACT,
    ACTTemporalEnsembler,
    ACTEncoder,
    ACTEncoderLayer,
    ACTDecoder,
    ACTDecoderLayer,
]:
    print("=" * 80)
    describe_object(obj)


对象: <class 'lerobot.policies.act.modeling_act.ACTPolicy'>
模块: lerobot.policies.act.modeling_act
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\modeling_act.py
起始行: 42
签名: (config: lerobot.policies.act.configuration_act.ACTConfig, **kwargs)
对象: <class 'lerobot.policies.act.modeling_act.ACT'>
模块: lerobot.policies.act.modeling_act
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\modeling_act.py
起始行: 258
签名: (config: lerobot.policies.act.configuration_act.ACTConfig)
对象: <class 'lerobot.policies.act.modeling_act.ACTTemporalEnsembler'>
模块: lerobot.policies.act.modeling_act
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\modeling_act.py
起始行: 167
签名: (temporal_ensemble_coeff: float, chunk_size: int) -> None
对象: <class 'lerobot.policies.act.modeling_act.ACTEncoder'>
模块: lerobot.policies.act.modeling_act
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\act\modeling_act.py
起始行: 5

## E1. ACTPolicy 的关键方法

```text
__init__()
    验证feature
    创建self.model = ACT(config)
    初始化动作队列或temporal ensembler

forward(batch)
    训练使用
    返回 loss 和 loss_dict

predict_action_chunk(batch)
    推理使用
    一次预测完整动作块

select_action(batch)
    推理使用
    每次只返回下一步动作
    内部管理动作队列

reset()
    每个episode开始时清空动作队列

get_optim_params()
    将backbone和其他参数分组，可设置不同学习率
```


In [11]:
print("ACTPolicy.__init__")
show_source(ACTPolicy.__init__, max_lines=100)


ACTPolicy.__init__
    def __init__(
        self,
        config: ACTConfig,
        **kwargs,
    ):
        """
        Args:
            config: Policy configuration class instance or None, in which case the default instantiation of
                    the configuration class is used.
        """
        super().__init__(config)
        config.validate_features()
        self.config = config

        self.model = ACT(config)

        if config.temporal_ensemble_coeff is not None:
            self.temporal_ensembler = ACTTemporalEnsembler(config.temporal_ensemble_coeff, config.chunk_size)

        self.reset()


In [12]:
print("ACTPolicy.forward")
show_source(ACTPolicy.forward, max_lines=140)


ACTPolicy.forward
    def forward(self, batch: dict[str, Tensor]) -> tuple[Tensor, dict]:
        """Run the batch through the model and compute the loss for training or validation."""
        if self.config.image_features:
            batch = dict(batch)  # shallow copy so that adding a key doesn't modify the original
            batch[OBS_IMAGES] = [batch[key] for key in self.config.image_features]

        actions_hat, (mu_hat, log_sigma_x2_hat) = self.model(batch)

        abs_err = F.l1_loss(batch[ACTION], actions_hat, reduction="none")
        valid_mask = ~batch["action_is_pad"].unsqueeze(-1)
        num_valid = valid_mask.sum() * abs_err.shape[-1]
        l1_loss = (abs_err * valid_mask).sum() / num_valid.clamp_min(1)

        loss_dict = {"l1_loss": l1_loss.item()}
        if self.config.use_vae and log_sigma_x2_hat is not None:
            # Calculate Dₖₗ(latent_pdf || standard_normal). Note: After computing the KL-divergence for
            # each dimension independently, we

In [13]:
print("ACTPolicy.predict_action_chunk")
show_source(ACTPolicy.predict_action_chunk, max_lines=100)


ACTPolicy.predict_action_chunk
    @torch.no_grad()
    def predict_action_chunk(self, batch: dict[str, Tensor]) -> Tensor:
        """Predict a chunk of actions given environment observations."""
        self.eval()

        if self.config.image_features:
            batch = dict(batch)  # shallow copy so that adding a key doesn't modify the original
            batch[OBS_IMAGES] = [batch[key] for key in self.config.image_features]

        actions = self.model(batch)[0]
        return actions


In [14]:
print("ACTPolicy.select_action")
show_source(ACTPolicy.select_action, max_lines=140)


ACTPolicy.select_action
    @torch.no_grad()
    def select_action(self, batch: dict[str, Tensor]) -> Tensor:
        """Select a single action given environment observations.

        This method wraps `select_actions` in order to return one action at a time for execution in the
        environment. It works by managing the actions in a queue and only calling `select_actions` when the
        queue is empty.
        """
        self.eval()  # keeping the policy in eval mode as it could be set to train mode while queue is consumed

        if self.config.temporal_ensemble_coeff is not None:
            actions = self.predict_action_chunk(batch)
            action = self.temporal_ensembler.update(actions)
            return action

        # Action queue logic for n_action_steps > 1. When the action_queue is depleted, populate it by
        # querying the policy.
        if len(self._action_queue) == 0:
            actions = self.predict_action_chunk(batch)[:, : self.config.n_action_ste

In [15]:
print("ACTPolicy.reset")
show_source(ACTPolicy.reset, max_lines=80)

print("\nACTPolicy.get_optim_params")
show_source(ACTPolicy.get_optim_params, max_lines=100)


ACTPolicy.reset
    def reset(self):
        """This should be called whenever the environment is reset."""
        if self.config.temporal_ensemble_coeff is not None:
            self.temporal_ensembler.reset()
        else:
            self._action_queue = deque([], maxlen=self.config.n_action_steps)

ACTPolicy.get_optim_params
    def get_optim_params(self) -> dict:
        # TODO(aliberts, rcadene): As of now, lr_backbone == lr
        # Should we remove this and just `return self.parameters()`?
        return [
            {
                "params": [
                    p
                    for n, p in self.named_parameters()
                    if not n.startswith("model.backbone") and p.requires_grad
                ]
            },
            {
                "params": [
                    p
                    for n, p in self.named_parameters()
                    if n.startswith("model.backbone") and p.requires_grad
                ],
                "lr": self.config.

## E2. `forward()`为什么只在训练时使用

训练 batch 包含专家动作：

```text
observation.image
observation.state
action
action_is_pad
```

流程：

```text
ACT预测 actions_hat
        ↓
与专家 action 比较
        ↓
屏蔽action_is_pad
        ↓
L1 loss
        +
kl_weight × KL loss
```

推理时没有专家 action，因此调用：

```python
predict_action_chunk(observation)
```

或：

```python
select_action(observation)
```


# F. 阅读 ACT 神经网络源码

`ACT.forward()` 是算法结构的核心。


In [16]:
print("ACT.__init__")
show_source(ACT.__init__, max_lines=220)


ACT.__init__
    def __init__(self, config: ACTConfig):
        # BERT style VAE encoder with input tokens [cls, robot_state, *action_sequence].
        # The cls token forms parameters of the latent's distribution (like this [*means, *log_variances]).
        super().__init__()
        self.config = config

        if self.config.use_vae:
            self.vae_encoder = ACTEncoder(config, is_vae_encoder=True)
            self.vae_encoder_cls_embed = nn.Embedding(1, config.dim_model)
            # Projection layer for joint-space configuration to hidden dimension.
            if self.config.robot_state_feature:
                self.vae_encoder_robot_state_input_proj = nn.Linear(
                    self.config.robot_state_feature.shape[0], config.dim_model
                )
            # Projection layer for action (joint-space target) to hidden dimension.
            self.vae_encoder_action_input_proj = nn.Linear(
                self.config.action_feature.shape[0],
                con

In [17]:
print("ACT.forward")
show_source(ACT.forward, max_lines=260)


ACT.forward
    def forward(self, batch: dict[str, Tensor]) -> tuple[Tensor, tuple[Tensor, Tensor] | tuple[None, None]]:
        """A forward pass through the Action Chunking Transformer (with optional VAE encoder).

        `batch` should have the following structure:
        {
            [robot_state_feature] (optional): (B, state_dim) batch of robot states.

            [image_features]: (B, n_cameras, C, H, W) batch of images.
                AND/OR
            [env_state_feature]: (B, env_dim) batch of environment states.

            [action_feature] (optional, only if training with VAE): (B, chunk_size, action dim) batch of actions.
        }

        Returns:
            (B, chunk_size, action_dim) batch of action sequences
            Tuple containing the latent PDF's parameters (mean, log(σ²)) both as (B, L) tensors where L is the
            latent dimension.
        """
        if self.config.use_vae and self.training:
            assert ACTION in batch, (
                

阅读 `ACT.forward()` 时建立以下数据流：

```text
训练时的专家action chunk
    ↓ action projection
CVAE Encoder
    ↓
mu, log_sigma_x2
    ↓ 重参数化
latent z
```

然后：

```text
latent token
+ robot state token
+ image feature tokens
    ↓
Transformer Encoder
    ↓
Transformer Decoder
+ chunk_size个learnable position query
    ↓
action_head
    ↓
(B, chunk_size, action_dim)
```

推理时没有专家动作：

```text
latent z = 0
```

这也是训练与推理路径的重要区别。


# G. LeRobot 工厂：数据集如何自动适配 ACT

不要手动猜测：

```python
input_features
output_features
```

LeRobot 官方训练管线通过：

```python
make_policy(config, ds_meta=metadata)
```

自动完成：

```text
metadata.features
    ↓
dataset_to_policy_features()
    ↓
FeatureType.VISUAL / STATE / ACTION
    ↓
ACTConfig.input_features
ACTConfig.output_features
    ↓
ACTPolicy
```


In [18]:
from lerobot.policies import make_policy, make_pre_post_processors

describe_object(make_policy)
show_source(make_policy, max_lines=220)


对象: <function make_policy at 0x0000019AE61BAAC0>
模块: lerobot.policies.factory
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\factory.py
起始行: 240
签名: (cfg: 'PreTrainedConfig', ds_meta: 'LeRobotDatasetMetadata | None' = None, env_cfg: 'EnvConfig | None' = None, rename_map: 'dict[str, str] | None' = None) -> 'PreTrainedPolicy'
def make_policy(
    cfg: PreTrainedConfig,
    ds_meta: LeRobotDatasetMetadata | None = None,
    env_cfg: EnvConfig | None = None,
    rename_map: dict[str, str] | None = None,
) -> PreTrainedPolicy:
    """
    Instantiate a policy model.

    This factory function handles the logic of creating a policy, which requires
    determining the input and output feature shapes. These shapes can be derived
    either from a `LeRobotDatasetMetadata` object or an `EnvConfig` object. The function
    can either initialize a new policy from scratch or load a pretrained one.

    Args:
        cfg: The configuration for the policy to be created. If

In [19]:
describe_object(make_pre_post_processors)
show_source(make_pre_post_processors, max_lines=180)


对象: <function make_pre_post_processors at 0x0000019AE6185260>
模块: lerobot.policies.factory
源码文件: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\policies\factory.py
起始行: 150
签名: (policy_cfg: 'PreTrainedConfig', pretrained_path: 'str | None' = None, pretrained_revision: 'str | None' = None, **kwargs: 'Unpack[ProcessorConfigKwargs]') -> 'tuple[PolicyProcessorPipeline[dict[str, Any], dict[str, Any]], PolicyProcessorPipeline[PolicyAction, PolicyAction]]'
def make_pre_post_processors(
    policy_cfg: PreTrainedConfig,
    pretrained_path: str | None = None,
    pretrained_revision: str | None = None,
    **kwargs: Unpack[ProcessorConfigKwargs],
) -> tuple[
    PolicyProcessorPipeline[dict[str, Any], dict[str, Any]],
    PolicyProcessorPipeline[PolicyAction, PolicyAction],
]:
    """
    Create or load pre- and post-processor pipelines for a given policy.

    This function acts as a factory. It can either load existing processor pipelines
    from a pretrained path or create new

# H. 加载 PushT 数据集并分析 ACT 输入输出


In [20]:
from lerobot.datasets.lerobot_dataset import (
    LeRobotDataset,
    LeRobotDatasetMetadata,
)

DATASET_ID = "lerobot/pusht"

metadata = LeRobotDatasetMetadata(DATASET_ID)

print("repo_id:", metadata.repo_id)
print("root:", metadata.root)
print("fps:", metadata.fps)
print("episodes:", metadata.total_episodes)
print("frames:", metadata.total_frames)

print("\nfeatures:")
for key, feature in metadata.features.items():
    print(key, feature)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 532.73it/s]


repo_id: lerobot/pusht
root: D:\Desktop\robot\datasets\lerobot\hub\datasets--lerobot--pusht\snapshots\b1c3ecbae7f244acc039a3dbc255a00dad1372b9
fps: 10
episodes: 206
frames: 25650

features:
observation.image {'dtype': 'video', 'shape': (96, 96, 3), 'names': ['height', 'width', 'channel'], 'video_info': {'video.fps': 10.0, 'video.codec': 'av1', 'video.pix_fmt': 'yuv420p', 'video.is_depth_map': False, 'has_audio': False}}
observation.state {'dtype': 'float32', 'shape': (2,), 'names': {'motors': ['motor_0', 'motor_1']}, 'fps': 10.0}
action {'dtype': 'float32', 'shape': (2,), 'names': {'motors': ['motor_0', 'motor_1']}, 'fps': 10.0}
episode_index {'dtype': 'int64', 'shape': (1,), 'names': None, 'fps': 10.0}
frame_index {'dtype': 'int64', 'shape': (1,), 'names': None, 'fps': 10.0}
timestamp {'dtype': 'float32', 'shape': (1,), 'names': None, 'fps': 10.0}
next.reward {'dtype': 'float32', 'shape': (1,), 'names': None, 'fps': 10.0}
next.done {'dtype': 'bool', 'shape': (1,), 'names': None, 'fps'

PushT 的策略相关字段应为：

```text
observation.image
    当前画面

observation.state
    圆形推杆当前二维位置

action
    圆形推杆二维目标位置
```

ACT模型输入：

```text
image + state
```

监督目标：

```text
未来action chunk
```


In [21]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

act_config = ACTConfig(
    device=DEVICE,
    push_to_hub=False,

    chunk_size=100,
    n_action_steps=100,

    use_amp=False,
    vision_backbone="resnet18",
    use_vae=True,
)

policy = make_policy(
    cfg=act_config,
    ds_meta=metadata,
)

print("Policy class:", type(policy))
print("Policy device:", next(policy.parameters()).device)

print("\n自动生成的input_features:")
for key, feature in policy.config.input_features.items():
    print(" ", key, feature)

print("\n自动生成的output_features:")
for key, feature in policy.config.output_features.items():
    print(" ", key, feature)


Policy class: <class 'lerobot.policies.act.modeling_act.ACTPolicy'>
Policy device: cuda:0

自动生成的input_features:
  observation.image PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 96, 96))
  observation.state PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(2,))

自动生成的output_features:
  action PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(2,))


# I. 展示 ACT 模型文件架构和模块架构

文件架构描述“代码在哪里”。

模块架构描述“模型运行时包含哪些子网络”。


In [22]:
print(policy)


ACTPolicy(
  (model): ACT(
    (vae_encoder): ACTEncoder(
      (layers): ModuleList(
        (0-3): 4 x ACTEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=3200, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=3200, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): Identity()
    )
    (vae_encoder_cls_embed): Embedding(1, 512)
    (vae_encoder_robot_state_input_proj): Linear(in_features=2, out_features=512, bias=True)
    (vae_encoder_action_input_proj): Linear(in_features=2, out_features=512, bias

In [23]:
def module_parameter_table(module: torch.nn.Module, max_depth: int = 2):
    rows = []

    for name, submodule in module.named_modules():
        if name == "":
            continue

        depth = name.count(".") + 1
        if depth > max_depth:
            continue

        own_params = sum(
            parameter.numel()
            for parameter in submodule.parameters(recurse=False)
        )

        rows.append(
            {
                "module": name,
                "class": type(submodule).__name__,
                "own_parameters": own_params,
            }
        )

    return rows


rows = module_parameter_table(policy, max_depth=3)

for row in rows:
    print(
        f"{row['module']:<55}"
        f"{row['class']:<32}"
        f"{row['own_parameters']:>12,}"
    )

print("\n总参数量:", f"{sum(p.numel() for p in policy.parameters()):,}")


model                                                  ACT                                        0
model.vae_encoder                                      ACTEncoder                                 0
model.vae_encoder.layers                               ModuleList                                 0
model.vae_encoder.norm                                 Identity                                   0
model.vae_encoder_cls_embed                            Embedding                                512
model.vae_encoder_robot_state_input_proj               Linear                                 1,536
model.vae_encoder_action_input_proj                    Linear                                 1,536
model.vae_encoder_latent_output_proj                   Linear                                32,832
model.backbone                                         IntermediateLayerGetter                    0
model.backbone.conv1                                   Conv2d                                 9,408


重点子模块通常包括：

```text
policy
└── model
    ├── backbone
    │   └── ResNet18视觉编码器
    ├── vae_encoder
    ├── encoder
    │   └── ACTEncoderLayer × n_encoder_layers
    ├── decoder
    │   └── ACTDecoderLayer × n_decoder_layers
    ├── encoder_*_input_proj
    ├── decoder_pos_embed
    └── action_head
```

以实际 `print(policy)` 输出为准。


# J. 将 PushT 单步 action 组织成 ACT action chunk

普通数据集：

```text
action shape = (2,)
```

ACT数据集：

```text
action shape = (100, 2)
```

由：

```python
policy.config.action_delta_indices
```

决定。


In [24]:
single_step_dataset = LeRobotDataset(DATASET_ID)
single_step_sample = single_step_dataset[0]

print("单步action:", single_step_sample["action"].shape)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 665.79it/s]


单步action: torch.Size([2])


In [25]:
action_delta_timestamps = [
    frame_offset / metadata.fps
    for frame_offset in policy.config.action_delta_indices
]

delta_timestamps = {
    "action": action_delta_timestamps,
}

act_dataset = LeRobotDataset(
    DATASET_ID,
    delta_timestamps=delta_timestamps,
)

act_sample = act_dataset[0]

print("ACT action chunk:", act_sample["action"].shape)
print("action_is_pad:", act_sample.get("action_is_pad").shape)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 665.23it/s]

ACT action chunk: torch.Size([100, 2])
action_is_pad: torch.Size([100])


`action_is_pad` 的作用：

当某个 frame 已经靠近 episode 结尾，未来不足100步时，LeRobot会补齐动作。

模型计算loss时必须屏蔽这些补齐位置，否则会把无效数据当作专家动作学习。


# K. DataLoader与预处理器


In [27]:
from torch.utils.data import DataLoader

loader = DataLoader(
    act_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE == "cuda"),
    drop_last=True,
)

raw_batch = next(iter(loader))

print("image:", raw_batch["observation.image"].shape)
print("state:", raw_batch["observation.state"].shape)
print("action:", raw_batch["action"].shape)
print("action_is_pad:", raw_batch["action_is_pad"].shape)


image: torch.Size([4, 3, 96, 96])
state: torch.Size([4, 2])
action: torch.Size([4, 100, 2])
action_is_pad: torch.Size([4, 100])


In [28]:
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config,
    dataset_stats=metadata.stats,
    dataset_meta=metadata,
)

print("Preprocessor:")
print(preprocessor)

print("\nPostprocessor:")
print(postprocessor)


Preprocessor:
DataProcessorPipeline(name='policy_preprocessor', steps=4: [RenameObservationsProcessorStep, AddBatchDimensionProcessorStep, ..., NormalizerProcessorStep])

Postprocessor:
DataProcessorPipeline(name='policy_postprocessor', steps=2: [UnnormalizerProcessorStep, DeviceProcessorStep])


In [29]:
processed_batch = preprocessor(raw_batch)

for key in [
    "observation.image",
    "observation.state",
    "action",
    "action_is_pad",
]:
    value = processed_batch.get(key)

    if value is not None:
        print(
            key,
            "shape=", getattr(value, "shape", None),
            "dtype=", getattr(value, "dtype", None),
            "device=", getattr(value, "device", None),
        )


observation.image shape= torch.Size([4, 3, 96, 96]) dtype= torch.float32 device= cuda:0
observation.state shape= torch.Size([4, 2]) dtype= torch.float32 device= cuda:0
action shape= torch.Size([4, 100, 2]) dtype= torch.float32 device= cuda:0
action_is_pad shape= torch.Size([4, 100]) dtype= torch.bool device= cuda:0


预处理器通常负责：

- 图像/状态/action归一化；
- 将Tensor移动到策略device；
- 处理策略约定的数据格式。

后处理器通常负责：

- 将归一化动作恢复到原始动作尺度。


# L. 手动执行一次真实 ACT 训练更新

这一节不调用官方训练器，目的是看清：

```text
forward
→ loss
→ backward
→ optimizer.step
```


In [56]:
policy.train()

loss, loss_dict = policy.forward(processed_batch)

print("loss:", float(loss.detach().cpu()))
print("loss_dict:", loss_dict)


loss: 7.638935565948486
loss_dict: {'l1_loss': 0.6858415007591248, 'kld_loss': 0.6953094005584717}


In [57]:
optimizer = torch.optim.AdamW(
    policy.get_optim_params(),
    lr=policy.config.optimizer_lr,
    weight_decay=policy.config.optimizer_weight_decay,
)

optimizer.zero_grad(set_to_none=True)
loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    policy.parameters(),
    max_norm=10.0,
)

optimizer.step()

print("完成一次参数更新")
print("gradient norm:", float(grad_norm.detach().cpu()))


完成一次参数更新
gradient norm: 291.54254150390625


这一格是真实ACT训练，不是简化线性模型。

但只训练一个batch没有实用性能，它的意义是完全掌握训练器内部最核心的更新过程。


# M. 阅读 LeRobot 官方训练脚本

下面动态定位：

```text
lerobot/scripts/lerobot_train.py
```

并列出它定义的函数。


In [58]:
import lerobot.scripts.lerobot_train as train_module

print("训练脚本:", inspect.getsourcefile(train_module))

train_functions = [
    (name, obj)
    for name, obj in inspect.getmembers(train_module, inspect.isfunction)
    if obj.__module__ == train_module.__name__
]

for name, obj in train_functions:
    try:
        signature = inspect.signature(obj)
    except Exception:
        signature = "<无法读取>"

    print(f"{name}{signature}")


训练脚本: d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\lerobot\scripts\lerobot_train.py
_dataloader_worker_kwargs(cfg: lerobot.configs.train.TrainPipelineConfig) -> dict[str, typing.Any]
_make_eval_envs(cfg: lerobot.configs.train.TrainPipelineConfig) -> collections.abc.Iterator[dict[str, dict[int, typing.Any]]]
_remote_target_in_argv() -> bool
main()
train(cfg: lerobot.configs.train.TrainPipelineConfig, accelerator: 'Accelerator | None' = None)
update_policy(train_metrics: lerobot.utils.logging_utils.MetricsTracker, policy: lerobot.policies.pretrained.PreTrainedPolicy, batch: Any, optimizer: torch.optim.optimizer.Optimizer, grad_clip_norm: float, accelerator: 'Accelerator', lr_scheduler=None, lock=None, sample_weighter=None) -> tuple[lerobot.utils.logging_utils.MetricsTracker, dict | None]


在训练脚本里重点寻找以下调用关系：

```text
train(cfg)
    ↓
make_train_eval_datasets()
    ↓
make_policy()
    ↓
make_pre_post_processors()
    ↓
make_optimizer_and_scheduler()
    ↓
DataLoader
    ↓
update_policy()
    ↓
save_checkpoint()
```

具体函数名可能随小版本变化；以上一格显示的实际结果为准。


In [59]:
if hasattr(train_module, "train"):
    show_source(train_module.train, max_lines=320)
else:
    print("当前版本未找到train()，请查看上一格函数列表。")


@parser.wrap()
def train(cfg: TrainPipelineConfig, accelerator: "Accelerator | None" = None):
    """
    Main function to train a policy.

    This function orchestrates the entire training pipeline, including:
    - Setting up logging, seeding, and device configuration.
    - Creating the dataset, evaluation environment (if applicable), policy, and optimizer.
    - Handling resumption from a checkpoint.
    - Running the main training loop, which involves fetching data batches and calling `update_policy`.
    - Periodically logging metrics, saving model checkpoints, and evaluating the policy.
    - Pushing the final trained model to the Hugging Face Hub if configured.

    Args:
        cfg: A `TrainPipelineConfig` object containing all training configurations.
        accelerator: Optional Accelerator instance. If None, one will be created automatically.
    """
    if cfg.job.is_remote:
        return submit_to_hf(cfg)

    require_package("accelerate", extra="training")
    from a

# N. 官方训练器：100 step 冒烟复现

冒烟训练目的：

- 验证官方数据管线；
- 验证GPU forward/backward；
- 验证checkpoint保存；
- 验证训练和推理文件完整。

它不是性能训练。


In [60]:
PROJECT_ROOT = Path(
    r"D:\Desktop\robot\workspace\embodied_learning"
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SMOKE_OUTPUT_DIR = OUTPUT_ROOT / f"05A_act_pusht_smoke_{run_stamp}"

train_entry = shutil.which("lerobot-train")

if train_entry:
    command_prefix = [train_entry]
else:
    command_prefix = [
        sys.executable,
        "-m",
        "lerobot.scripts.lerobot_train",
    ]

smoke_command = command_prefix + [
    "--dataset.repo_id=lerobot/pusht",
    "--policy.type=act",
    f"--output_dir={SMOKE_OUTPUT_DIR}",
    "--job_name=05A_act_pusht_smoke",

    f"--policy.device={DEVICE}",
    "--policy.push_to_hub=false",
    "--wandb.enable=false",

    "--batch_size=4",
    "--num_workers=0",

    "--steps=100",
    "--log_freq=10",

    "--save_checkpoint=true",
    "--save_freq=100",

    "--policy.chunk_size=100",
    "--policy.n_action_steps=100",
    "--policy.use_amp=false",
]

print("训练命令：")
print(subprocess.list2cmdline(smoke_command))

print("\n输出目录：")
print(SMOKE_OUTPUT_DIR)


训练命令：
D:\Desktop\robot\envs\lerobot-win\Scripts\lerobot-train.EXE --dataset.repo_id=lerobot/pusht --policy.type=act --output_dir=D:\Desktop\robot\workspace\embodied_learning\outputs\05A_act_pusht_smoke_20260814_133744 --job_name=05A_act_pusht_smoke --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false --batch_size=4 --num_workers=0 --steps=100 --log_freq=10 --save_checkpoint=true --save_freq=100 --policy.chunk_size=100 --policy.n_action_steps=100 --policy.use_amp=false

输出目录：
D:\Desktop\robot\workspace\embodied_learning\outputs\05A_act_pusht_smoke_20260814_133744


In [61]:
RUN_SMOKE_TRAIN = False

if RUN_SMOKE_TRAIN:
    subprocess.run(
        smoke_command,
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("未启动。确认命令后，把 RUN_SMOKE_TRAIN 改成 True。")


未启动。确认命令后，把 RUN_SMOKE_TRAIN 改成 True。


# O. 正式基线训练

建议流程：

```text
100 step 冒烟
→ 5,000 step检查loss和速度
→ 80,000或100,000 step形成初步基线
→ PushT closed-loop评测
```

RTX4060首次可以从：

```text
batch_size=8
steps=80_000
```

开始。发生显存不足时将batch减为4。


In [ ]:
full_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
FULL_OUTPUT_DIR = OUTPUT_ROOT / f"05A_act_pusht_full_{full_stamp}"

full_command = command_prefix + [
    "--dataset.repo_id=lerobot/pusht",
    "--policy.type=act",
    f"--output_dir={FULL_OUTPUT_DIR}",
    "--job_name=05A_act_pusht_full",

    f"--policy.device={DEVICE}",
    "--policy.push_to_hub=false",
    "--wandb.enable=false",

    "--batch_size=8",
    "--num_workers=0",

    "--steps=80000",
    "--log_freq=100",

    "--save_checkpoint=true",
    "--save_freq=10000",

    "--policy.chunk_size=100",
    "--policy.n_action_steps=100",
    "--policy.use_amp=false",
]

print(subprocess.list2cmdline(full_command))


In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    subprocess.run(
        full_command,
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("未启动正式训练。")


# P. 查看本地 checkpoint 文件架构


In [ ]:
def find_pretrained_model_dirs(root: Path) -> list[Path]:
    if not root.exists():
        return []

    return sorted(
        path
        for path in root.rglob("pretrained_model")
        if path.is_dir()
    )


def print_checkpoint_tree(root: Path, max_depth: int = 4) -> None:
    if not root.exists():
        print("目录不存在:", root)
        return

    print_tree(root, max_depth=max_depth)


print("冒烟输出目录:", SMOKE_OUTPUT_DIR)
print_checkpoint_tree(SMOKE_OUTPUT_DIR, max_depth=5)

local_model_dirs = find_pretrained_model_dirs(SMOKE_OUTPUT_DIR)

print("\n找到的pretrained_model:")
for item in local_model_dirs:
    print(item)


一个可加载的 LeRobot checkpoint 通常包含：

```text
pretrained_model/
├── config.json
├── model.safetensors
├── train_config.json
├── policy_preprocessor.json
├── policy_postprocessor.json
└── 对应processor权重文件
```

文件名以你实际训练结果为准。

其中：

```text
config.json
    ACT模型结构与输入输出feature

model.safetensors
    神经网络权重

train_config.json
    完整训练配置

policy_preprocessor*
    输入归一化与处理逻辑

policy_postprocessor*
    动作反归一化与输出处理逻辑
```


# Q. 加载本地训练后的 ACT


In [ ]:
LOCAL_MODEL_DIR = (
    local_model_dirs[-1]
    if local_model_dirs
    else None
)

if LOCAL_MODEL_DIR is None:
    print("没有找到本地checkpoint，请先完成冒烟训练。")
else:
    local_policy = ACTPolicy.from_pretrained(
        str(LOCAL_MODEL_DIR)
    )
    local_policy.to(DEVICE)
    local_policy.eval()

    local_preprocessor, local_postprocessor = make_pre_post_processors(
        policy_cfg=local_policy.config,
        pretrained_path=str(LOCAL_MODEL_DIR),
        dataset_stats=metadata.stats,
        dataset_meta=metadata,
    )

    print("本地模型加载成功")
    print("device:", next(local_policy.parameters()).device)
    print("chunk_size:", local_policy.config.chunk_size)


# R. Hub上是否存在PushT ACT预训练模型

这格实时查询：

- 是否存在官方 `lerobot/act_pusht`；
- 社区模型是否仍可访问；
- 模型仓库有哪些文件。


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

MODEL_CANDIDATES = [
    "lerobot/act_pusht",
    "aadarshram/act_pusht",
    "Lemon-03/ACT_PushT_test",
    "vvrs/act-pusht",
]

for model_id in MODEL_CANDIDATES:
    try:
        info = api.model_info(model_id)
        print(
            "[存在]",
            model_id,
            "downloads=", getattr(info, "downloads", None),
            "last_modified=", getattr(info, "last_modified", None),
        )
    except Exception as exc:
        print("[未确认]", model_id, type(exc).__name__, str(exc)[:120])


根据目前检索结果：

- 没有确认到 LeRobot 官方组织公开的 `lerobot/act_pusht`；
- `aadarshram/act_pusht` 是社区模型，模型卡称其使用 `lerobot/pusht` 训练80,000步；
- `Lemon-03/ACT_PushT_test` 也是社区基线，并公开了训练配置；
- 社区checkpoint只能作为对照，不代表官方性能。

默认选择：

```text
aadarshram/act_pusht
```


In [ ]:
PRETRAINED_MODEL_ID = "aadarshram/act_pusht"

repo_files = api.list_repo_files(
    PRETRAINED_MODEL_ID,
    repo_type="model",
)

print(PRETRAINED_MODEL_ID)
for filename in repo_files:
    print("├──", filename)


预期可看到类似：

```text
README.md
config.json
model.safetensors
train_config.json
policy_preprocessor.json
policy_postprocessor.json
policy_preprocessor_step_*_normalizer_processor.safetensors
policy_postprocessor_step_*_unnormalizer_processor.safetensors
```

这就是一个完整可推理LeRobot策略仓库的文件架构。


# S. 下载并加载社区预训练 PushT ACT

该步骤会下载约数百MB权重，默认关闭。


In [ ]:
from huggingface_hub import snapshot_download

RUN_PRETRAINED_DOWNLOAD = False

if RUN_PRETRAINED_DOWNLOAD:
    pretrained_snapshot = Path(
        snapshot_download(
            repo_id=PRETRAINED_MODEL_ID,
            repo_type="model",
        )
    )

    print("下载目录:", pretrained_snapshot)
    print_tree(pretrained_snapshot, max_depth=3)
else:
    pretrained_snapshot = None
    print("未下载。准备好后将 RUN_PRETRAINED_DOWNLOAD 改为 True。")


In [ ]:
if pretrained_snapshot is None:
    print("请先下载预训练模型。")
else:
    reference_policy = ACTPolicy.from_pretrained(
        str(pretrained_snapshot)
    )
    reference_policy.to(DEVICE)
    reference_policy.eval()

    reference_preprocessor, reference_postprocessor = make_pre_post_processors(
        policy_cfg=reference_policy.config,
        pretrained_path=str(pretrained_snapshot),
        dataset_stats=metadata.stats,
        dataset_meta=metadata,
    )

    print("社区预训练ACT加载成功")
    print("device:", next(reference_policy.parameters()).device)
    print("chunk_size:", reference_policy.config.chunk_size)
    print("n_action_steps:", reference_policy.config.n_action_steps)
    print("parameter count:", sum(p.numel() for p in reference_policy.parameters()))


# T. 离线推理：Dataset observation → ACT action

离线推理只能回答：

> 给定数据集中的某个观察，模型输出了什么动作？

它不能证明任务是否成功；成功率必须通过PushT环境闭环rollout测量。


In [ ]:
def make_inference_batch(
    sample: dict[str, Any],
    policy_config: ACTConfig,
) -> dict[str, Any]:
    batch = {}

    for key in policy_config.input_features:
        value = sample[key]

        if torch.is_tensor(value):
            batch[key] = value.unsqueeze(0)
        else:
            batch[key] = [value]

    return batch


raw_inference_batch = make_inference_batch(
    single_step_sample,
    policy.config,
)

for key, value in raw_inference_batch.items():
    print(key, getattr(value, "shape", None), type(value))


In [ ]:
policy.eval()
policy.reset()

processed_obs = preprocessor(raw_inference_batch)

with torch.no_grad():
    predicted_chunk_normalized = policy.predict_action_chunk(
        processed_obs
    )
    first_action_normalized = policy.select_action(
        processed_obs
    )

print("predicted chunk:", predicted_chunk_normalized.shape)
print("first action:", first_action_normalized.shape)

predicted_chunk = postprocessor(predicted_chunk_normalized)
first_action = postprocessor(first_action_normalized)

print("反归一化chunk:", getattr(predicted_chunk, "shape", None))
print("反归一化first action:", getattr(first_action, "shape", None))
print("专家第一步action:", single_step_sample["action"])


## `predict_action_chunk()`与`select_action()`的区别

```text
predict_action_chunk()
    每次调用模型
    返回(B, chunk_size, action_dim)

select_action()
    每个控制周期调用
    返回(B, action_dim)
    内部使用动作队列
    队列为空时才重新预测chunk
```

因此真实机器人控制循环通常调用：

```python
policy.select_action(observation)
```

而不是每一步手动处理整个chunk。


# U. 在相同观察上比较本地模型和社区预训练模型

这里比较的是：

- 配置；
- 参数量；
- 同一输入下的动作输出；
- 与当前专家动作的单样本误差。

它不是最终性能评测。


In [ ]:
def summarize_policy(name: str, current_policy: ACTPolicy) -> dict[str, Any]:
    return {
        "name": name,
        "class": type(current_policy).__name__,
        "parameters": sum(p.numel() for p in current_policy.parameters()),
        "chunk_size": current_policy.config.chunk_size,
        "n_action_steps": current_policy.config.n_action_steps,
        "dim_model": current_policy.config.dim_model,
        "encoder_layers": current_policy.config.n_encoder_layers,
        "decoder_layers": current_policy.config.n_decoder_layers,
        "use_vae": current_policy.config.use_vae,
        "latent_dim": current_policy.config.latent_dim,
        "kl_weight": current_policy.config.kl_weight,
    }


summaries = [
    summarize_policy("当前ACT", policy),
]

if "reference_policy" in globals():
    summaries.append(
        summarize_policy(
            PRETRAINED_MODEL_ID,
            reference_policy,
        )
    )

for summary in summaries:
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
if "reference_policy" not in globals():
    print("尚未加载社区预训练模型。")
else:
    reference_policy.reset()

    reference_processed_obs = reference_preprocessor(
        raw_inference_batch
    )

    with torch.no_grad():
        reference_action_normalized = reference_policy.select_action(
            reference_processed_obs
        )

    reference_action = reference_postprocessor(
        reference_action_normalized
    )

    expert_action = single_step_sample["action"].unsqueeze(0)

    print("社区模型action:")
    print(reference_action)

    print("\n专家action:")
    print(expert_action)

    if torch.is_tensor(reference_action):
        error = torch.nn.functional.l1_loss(
            reference_action.cpu(),
            expert_action.cpu(),
        )
        print("\n单样本L1误差:", float(error))


# V. PushT闭环推理与评测

真正的性能比较必须运行：

```text
env.reset()
→ observation
→ policy.select_action()
→ env.step(action)
→ next observation
→ 直到episode结束
```

LeRobot统一评测入口：

```powershell
lerobot-eval
```

首次使用需要PushT环境依赖。可在PowerShell安装：

```powershell
python -m pip install gym-pusht
```


In [ ]:
try:
    import gym_pusht
    print("gym_pusht:", gym_pusht.__file__)
except Exception as exc:
    print("gym_pusht未就绪:", exc)
    print('PowerShell安装：python -m pip install gym-pusht')


In [ ]:
EVAL_MODEL_PATH = (
    str(LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR is not None
    else PRETRAINED_MODEL_ID
)

eval_entry = shutil.which("lerobot-eval")

if eval_entry:
    eval_prefix = [eval_entry]
else:
    eval_prefix = [
        sys.executable,
        "-m",
        "lerobot.scripts.lerobot_eval",
    ]

eval_output = OUTPUT_ROOT / (
    "05A_eval_"
    + EVAL_MODEL_PATH.replace("/", "_").replace("\\", "_")
)

eval_command = eval_prefix + [
    f"--policy.path={EVAL_MODEL_PATH}",
    "--env.type=pusht",
    "--eval.n_episodes=10",
    "--eval.batch_size=1",
    f"--output_dir={eval_output}",
    "--policy.device=cuda",
    "--policy.use_amp=false",
]

print(subprocess.list2cmdline(eval_command))


In [ ]:
RUN_CLOSED_LOOP_EVAL = False

if RUN_CLOSED_LOOP_EVAL:
    subprocess.run(
        eval_command,
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("未启动闭环评测。")


闭环评测应重点查看：

```text
avg_max_reward
    episode中最高目标重叠度

pc_success
    成功率

avg_sum_reward
    累计奖励
```

比较实验必须保持：

- 同一个PushT环境版本；
- 相同随机种子；
- 相同episode数量；
- 相同成功判据；
- 相同预处理器；
- 相同动作执行策略。


# W. 训练、保存、加载、推理的完整调用图

```text
[训练]

LeRobotDatasetMetadata
        ↓
make_policy(ACTConfig, metadata)
        ↓
ACTPolicy
        └── ACT
            ├── ResNet backbone
            ├── VAE encoder
            ├── Transformer encoder
            ├── Transformer decoder
            └── action head

LeRobotDataset(delta_timestamps)
        ↓
DataLoader
        ↓
make_pre_post_processors
        ↓
ACTPolicy.forward(batch)
        ↓
L1 + KL loss
        ↓
backward
        ↓
optimizer.step
        ↓
save_checkpoint


[推理]

ACTPolicy.from_pretrained()
        ↓
make_pre_post_processors(pretrained_path)
        ↓
policy.reset()
        ↓
observation
        ↓
preprocessor
        ↓
policy.select_action()
        ├── queue为空
        │   └── predict_action_chunk()
        └── 从queue弹出一个action
        ↓
postprocessor
        ↓
env.step(action)
```


# X. 完成检查表

完成 05A 后应当能够独立解释：

- [ ] `ACTConfig`如何决定action chunk
- [ ] `make_policy()`如何从dataset metadata推断feature
- [ ] `ACTPolicy`和`ACT`的区别
- [ ] `forward()`为何需要专家action
- [ ] `predict_action_chunk()`输出什么
- [ ] `select_action()`为何维护动作队列
- [ ] `reset()`为什么每个episode都要调用
- [ ] `get_optim_params()`为什么把backbone单独分组
- [ ] `action_is_pad`如何参与loss
- [ ] L1 loss和KL loss在哪里计算
- [ ] 预处理器和后处理器为什么要随checkpoint保存
- [ ] 官方训练器内部的调用顺序
- [ ] checkpoint每个文件的作用
- [ ] 离线动作误差和闭环成功率的区别
- [ ] 官方模型与社区模型的区别

下一步：

```text
05B_ACT模型白盒搭建与逐层复现.ipynb
```

05B 将不再调用封装好的 `ACT` 网络，而是亲自实现：

```text
CVAE Encoder
ResNet feature projection
Transformer Encoder/Decoder
Action Query
Action Head
L1 + KL Loss
```

并与 05A 的官方实现逐层对照。
